# LeetCode 4047 — Minimum Operations to Make XOR of All Elements Zero (Hard)

Bài toán LeetCode (bài premium):

```python
class Solution:
    def minOperations(self, nums: List[int]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4047

Bài này là một ví dụ đẹp về **tìm bất biến (invariant)**: thay vì theo dõi cả mảng (quá nhiều trạng thái), ta chỉ cần theo dõi **một con số** — tổng XOR — và bài toán biến thành **BFS trên đồ thị 2048 đỉnh**.

---

## Sau notebook này, bạn sẽ nắm được

1. Các tính chất của XOR: giao hoán, kết hợp, `x ^ x = 0`, `x ^ 0 = x`.
2. Phân tích một phép biến đổi để xem nó tác động lên **tổng XOR** thế nào.
3. Brute Force: BFS trên toàn bộ trạng thái mảng (chỉ dùng để kiểm chứng).
4. Quy về: "chọn ít giá trị nhất có XOR bằng tổng" — thử tập con, rồi BFS trên giá trị XOR.
5. Hai trường hợp `-1` và lý do bằng **đại số tuyến tính trên GF(2)** (độc lập tuyến tính).
6. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho mảng `nums` gồm các số nguyên dương. Một **thao tác**:

- chọn hai chỉ số `i != j` với `nums[i] != nums[j]`;
- thay **một trong hai** phần tử đó bằng `nums[i] ^ nums[j]`.

Tìm số thao tác **ít nhất** để XOR của toàn mảng bằng `0`. Nếu không thể, trả về `-1`.

---

## Ví dụ 1

```python
nums = [8, 1, 4, 8, 2]   ->   3
```

Tổng XOR ban đầu: `8 ^ 1 ^ 4 ^ 8 ^ 2 = 7`. Cần 3 thao tác (giải thích ở mục 4).

## Ví dụ 2

```python
nums = [1, 2, 3]   ->   0
```

`1 ^ 2 ^ 3 = 0` sẵn rồi.

## Ví dụ 3

```python
nums = [1, 2, 4]   ->   -1
```

Không cách nào đưa tổng XOR về 0.

---

## Ràng buộc

```text
2 <= nums.length <= 10^5
1 <= nums[i] <= 2000
```

# 2. Kiến thức nền tảng

## 2.1. Tính chất XOR

- `a ^ a = 0`, `a ^ 0 = a`.
- Giao hoán, kết hợp: thứ tự XOR không quan trọng.
- Vì vậy XOR một giá trị **hai lần** coi như không XOR.

In [1]:
from typing import List
from functools import reduce
from operator import xor

print(5 ^ 5, 5 ^ 0, (3 ^ 6) ^ 5 == 3 ^ (6 ^ 5))
print(reduce(xor, [8, 1, 4, 8, 2]))

0 5 True
7


## 2.2. Giá trị nhỏ ⇒ XOR nhỏ

`nums[i] <= 2000 < 2048 = 2^11` nên mọi XOR của các phần tử đều nằm trong `[0, 2047]`: chỉ có **2048** giá trị khả dĩ.

In [2]:
print(2000 .bit_length(), (1 << 11) - 1, 2000 ^ 1999, 1024 ^ 1023)

11 2047 31 2047


## 2.3. Thao tác tác động lên tổng XOR thế nào?

Gọi `T` là tổng XOR, giả sử thay `nums[i]` bằng `nums[i] ^ nums[j]` (giữ nguyên `nums[j]`):

```text
T_mới = T ^ nums[i] ^ (nums[i] ^ nums[j]) = T ^ nums[j]
```

**Mỗi thao tác XOR tổng với giá trị của phần tử ĐƯỢC GIỮ LẠI.** Đây là chìa khóa của cả bài.

In [3]:
def apply_op(arr, i, j):
    # thay arr[i] bằng arr[i] ^ arr[j], giữ arr[j]
    new = arr[:]
    new[i] = arr[i] ^ arr[j]
    return new


arr = [8, 1, 4, 8, 2]
T = reduce(xor, arr)
new = apply_op(arr, 0, 2)               # giữ arr[2] = 4
print(new, "T cũ =", T, "| T mới =", reduce(xor, new), "| T ^ 4 =", T ^ 4)

[12, 1, 4, 8, 2] T cũ = 7 | T mới = 3 | T ^ 4 = 3


# 3. Cách 1 — Brute Force: BFS trên toàn bộ trạng thái mảng

Mỗi trạng thái là cả mảng; mỗi cạnh là một thao tác hợp lệ. BFS tìm số thao tác ít nhất tới trạng thái có tổng XOR = 0. Số trạng thái bùng nổ rất nhanh — chỉ dùng với mảng nhỏ, giá trị nhỏ, và giới hạn độ sâu `cap`.

In [4]:
from collections import deque


def min_ops_bruteforce(nums: List[int], cap: int = 7) -> int:
    start = tuple(nums)
    if reduce(xor, start) == 0:
        return 0
    seen = {start}
    queue = deque([(start, 0)])
    while queue:
        state, d = queue.popleft()
        if d >= cap:
            continue
        n = len(state)
        for i in range(n):
            for j in range(n):
                if i != j and state[i] != state[j]:
                    nxt = list(state)
                    nxt[i] = state[i] ^ state[j]
                    nxt = tuple(nxt)
                    if nxt not in seen:
                        if reduce(xor, nxt) == 0:
                            return d + 1
                        seen.add(nxt)
                        queue.append((nxt, d + 1))
    return -1


print(min_ops_bruteforce([8, 1, 4, 8, 2]), min_ops_bruteforce([1, 2, 3]), min_ops_bruteforce([1, 2, 4]))

3 0 -1


(Thay `nums[j]` thay vì `nums[i]` chỉ là đổi vai `i`, `j` — vòng lặp đã duyệt mọi cặp có thứ tự nên đã bao gồm.)

---

# 4. Quy về bài "chọn ít giá trị nhất"

Từ mục 2.3: sau `d` thao tác giữ lại các giá trị `v1, v2, ..., vd`, tổng XOR là

```text
T ^ v1 ^ v2 ^ ... ^ vd
```

Ta muốn nó bằng 0 ⇔ `v1 ^ v2 ^ ... ^ vd = T`. Trực giác: dùng chính các **giá trị ban đầu** làm phần tử được giữ là rẻ nhất — tạo ra giá trị mới (dạng `a ^ b`) cũng tốn một thao tác, mà thao tác đó đã làm thay đổi tổng rồi, nên không tiết kiệm được gì. Kết quả (được kiểm chứng bằng brute force ở mục 9):

```text
nếu T == 0                        -> 0
nếu mọi phần tử bằng nhau         -> -1   (không thao tác nào hợp lệ)
d = số ít nhất các giá trị phân biệt trong nums có XOR = T
nếu d == len(nums)                -> -1   (xem mục 5)
ngược lại                         -> d
```

## Ví dụ 1

`nums = [8, 1, 4, 8, 2]`, `T = 7`. Giá trị phân biệt `{1, 2, 4, 8}`. Không giá trị nào bằng 7; không cặp nào XOR ra 7 (`1^2=3, 1^4=5, 2^4=6, ...^8 >= 8`); bộ ba `1 ^ 2 ^ 4 = 7` ✔. Vậy `d = 3 < 5` → **3**.

Một dãy thao tác cụ thể: dùng ô `nums[0] = 8` làm "chỗ nháp" bị thay, lần lượt giữ `1`, `4`, `2`:

| bước | giữ | mảng sau | tổng XOR |
|---:|---:|---|---:|
| 0 | | `[8, 1, 4, 8, 2]` | 7 |
| 1 | `1` (ô 1) | `[9, 1, 4, 8, 2]` | 6 |
| 2 | `4` (ô 2) | `[13, 1, 4, 8, 2]` | 2 |
| 3 | `2` (ô 4) | `[15, 1, 4, 8, 2]` | 0 |

In [5]:
arr = [8, 1, 4, 8, 2]
for keep in [1, 2, 4]:
    arr = apply_op(arr, 0, keep)
    print(arr, "tổng XOR =", reduce(xor, arr))

[9, 1, 4, 8, 2] tổng XOR = 6
[13, 1, 4, 8, 2] tổng XOR = 2
[15, 1, 4, 8, 2] tổng XOR = 0


# 5. Vì sao `d == len(nums)` thì không thể? — một chút đại số tuyến tính

Coi mỗi số là một **vector bit** (11 chiều) trên trường `GF(2)` = `{0, 1}` với phép cộng là XOR.

- `d == n` nghĩa là **phải dùng cả mảng** mới XOR ra `T`, tức không có tập con thật sự nào làm được. Khi đó các phần tử đôi một khác nhau và **không có tập con khác rỗng nào XOR bằng 0** (nếu có tập `Z` XOR bằng 0, thì "cả mảng bỏ `Z`" cũng XOR ra `T` với ít phần tử hơn). Nói cách khác: các phần tử **độc lập tuyến tính**.
- Thao tác `nums[i] ^= nums[j]` là **phép biến đổi sơ cấp** (cộng hàng `j` vào hàng `i`) — nó **giữ nguyên** tính độc lập tuyến tính (và có thể đảo ngược bằng cách làm lại lần nữa).
- Tổng XOR của các vector độc lập tuyến tính **không bao giờ bằng 0** (vì chính nó là một tổ hợp khác rỗng).

Vậy mảng mãi mãi độc lập, tổng XOR mãi khác 0 → `-1`. Ví dụ 3 `[1, 2, 4]` = `001, 010, 100` là một cơ sở.

In [6]:
def has_zero_subset(nums):
    # có tập con khác rỗng XOR = 0 ? (Gauss trên GF(2) với cơ sở XOR)
    basis = []
    for x in nums:
        for b in basis:
            x = min(x, x ^ b)
        if x == 0:
            return True
        basis.append(x)
    return False


for arr in [[1, 2, 4], [1, 2, 3], [8, 1, 4, 8, 2], [3, 5, 6, 9]]:
    print(arr, "có tập con XOR = 0:", has_zero_subset(arr))

[1, 2, 4] có tập con XOR = 0: False
[1, 2, 3] có tập con XOR = 0: True
[8, 1, 4, 8, 2] có tập con XOR = 0: True
[3, 5, 6, 9] có tập con XOR = 0: True


(Chiều ngược lại — khi `d < n` thì luôn làm được đúng `d` thao tác — cần một cách xây dựng hơi kỹ thuật: luôn tìm được một ô "nháp" ngoài các phần tử được giữ để làm chỗ bị thay, như bảng ở mục 4. Notebook kiểm chứng điều này bằng BFS trên toàn bộ trạng thái ở mục 9.)

# 6. Cách 2 — Thử tập con theo kích thước tăng dần

Tìm `d` bằng cách thử mọi tập con của các giá trị phân biệt, kích thước `1, 2, 3, ...`. Dễ hiểu nhưng **lũy thừa** — chỉ dùng khi ít giá trị phân biệt.

In [7]:
from itertools import combinations


def min_ops_subsets(nums: List[int]) -> int:
    T = reduce(xor, nums)
    if T == 0:
        return 0
    vals = sorted(set(nums))
    if len(vals) == 1:
        return -1
    for size in range(1, len(vals) + 1):
        for combo in combinations(vals, size):
            if reduce(xor, combo) == T:
                return -1 if size == len(nums) else size
    return -1


print(min_ops_subsets([8, 1, 4, 8, 2]), min_ops_subsets([1, 2, 3]), min_ops_subsets([1, 2, 4]))

3 0 -1


# 7. Cách 3 — BFS trên giá trị XOR (2048 đỉnh)

Đồ thị: đỉnh là các giá trị `0..2047`; từ đỉnh `u` có cạnh tới `u ^ v` với mỗi giá trị phân biệt `v`. Khoảng cách BFS từ `0` tới `T` = số giá trị ít nhất có XOR bằng `T` (dùng một giá trị hai lần thì triệt tiêu, BFS tự tránh vì không bao giờ ngắn hơn).

Độ phức tạp: `2048 × (số giá trị phân biệt) <= 2048 × 2000 ≈ 4·10^6`.

## Mô phỏng ví dụ 1 (`vals = {1, 2, 4, 8}`, `T = 7`)

| tầng | các đỉnh mới |
|---:|---|
| 0 | `0` |
| 1 | `1, 2, 4, 8` |
| 2 | `3, 5, 9, 6, 10, 12` |
| 3 | `7, 11, 13, 14` ← gặp `T = 7` |

In [8]:
def min_ops_bfs(nums: List[int], verbose: bool = False) -> int:
    T = reduce(xor, nums)
    if T == 0:
        return 0
    vals = set(nums)
    if len(vals) == 1:
        return -1
    dist = {0: 0}
    layer = [0]
    while layer:
        if verbose:
            print(f"tầng {dist[layer[0]]}: {layer}")
        nxt = []
        for u in layer:
            if u == T:
                return -1 if dist[u] == len(nums) else dist[u]
            for v in vals:
                w = u ^ v
                if w not in dist:
                    dist[w] = dist[u] + 1
                    nxt.append(w)
        layer = nxt
    return -1


print(min_ops_bfs([8, 1, 4, 8, 2], verbose=True))

tầng 0: [0]
tầng 1: [8, 1, 2, 4]
tầng 2: [9, 10, 12, 3, 5, 6]
tầng 3: [11, 13, 14, 7]
3


Vì `T` luôn là XOR của **cả mảng**, `T` luôn đến được (khoảng cách `<= số giá trị phân biệt`), nên `return -1` cuối hàm thực ra không bao giờ chạy.

# 8. Phiên bản đúng format LeetCode

In [9]:
from collections import deque
from typing import List


class Solution:
    def minOperations(self, nums: List[int]) -> int:
        total = 0
        for x in nums:
            total ^= x
        if total == 0:
            return 0
        vals = set(nums)
        if len(vals) == 1:
            return -1
        dist = [-1] * 2048
        dist[0] = 0
        queue = deque([0])
        while queue:
            u = queue.popleft()
            if u == total:
                return -1 if dist[u] == len(nums) else dist[u]
            for v in vals:
                w = u ^ v
                if dist[w] == -1:
                    dist[w] = dist[u] + 1
                    queue.append(w)
        return -1

In [10]:
sol = Solution()
tests = [
    ([8, 1, 4, 8, 2], 3, "ví dụ 1"),
    ([1, 2, 3], 0, "ví dụ 2"),
    ([1, 2, 4], -1, "ví dụ 3: cơ sở độc lập"),
    ([7, 7, 7], -1, "một giá trị, T != 0"),
    ([7, 7], 0, "một giá trị, T = 0"),
    ([1, 2, 4, 1, 1], 3, "d = số giá trị phân biệt < n"),
    ([5, 6, 9, 9], 2, "cần hai giá trị"),
]
for nums, expected, name in tests:
    result = sol.minOperations(nums)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:30} -> {result}")

PASS | ví dụ 1                        -> 3
PASS | ví dụ 2                        -> 0
PASS | ví dụ 3: cơ sở độc lập         -> -1
PASS | một giá trị, T != 0            -> -1
PASS | một giá trị, T = 0             -> 0
PASS | d = số giá trị phân biệt < n   -> 3
PASS | cần hai giá trị                -> 2


Test `[1, 2, 4, 1, 1]`: `T = 7`, cần đủ ba giá trị phân biệt `{1, 2, 4}`, nhưng `n = 5 > 3` — có hai ô `1` dư làm chỗ nháp → làm được bằng 3 thao tác.

# 9. Kiểm chứng bằng BFS trên toàn bộ trạng thái

In [11]:
import random

random.seed(4047)
mismatch = 0
for _ in range(1200):
    nums = [random.randint(1, 7) for _ in range(random.randint(2, 5))]
    expected = min_ops_bruteforce(nums)
    got = (sol.minOperations(nums), min_ops_bfs(nums), min_ops_subsets(nums))
    if any(g != expected for g in got):
        mismatch += 1
print("Số test lệch:", mismatch, "/ 1200")

Số test lệch: 0 / 1200


In [12]:
import time

big = [random.randint(1, 2000) for _ in range(10**5)]
start = time.time()
print(sol.minOperations(big), f"({time.time() - start:.2f} giây)")

1 (0.14 giây)


# 10. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. BFS trạng thái mảng | toàn bộ mảng là đỉnh | lũy thừa | lũy thừa |
| 2. Thử tập con | tổ hợp giá trị phân biệt | `O(2^D · D)` | `O(D)` |
| 3. BFS trên XOR | 2048 đỉnh, `D` cạnh mỗi đỉnh | `O(n + 2048 · D)` | `O(2048)` |

(`D` = số giá trị phân biệt `<= 2000`.)

# 11. Vì sao đúng? (tóm tắt)

1. Mỗi thao tác XOR tổng với phần tử được giữ (mục 2.3).
2. Cần `T ^ v1 ^ ... ^ vd = 0`; ít nhất là số giá trị ban đầu ít nhất có XOR = `T` (trực giác mục 4, kiểm chứng mục 9).
3. Không có thao tác nào khi mọi phần tử bằng nhau → `-1` (nếu `T != 0`).
4. `d == n` ⇔ mảng độc lập tuyến tính; thao tác giữ độc lập; tổng XOR của tập độc lập khác 0 → `-1` (mục 5).

# 12. Những lỗi dễ mắc

## Lỗi 1 — Quên trường hợp `d == len(nums)`

In [13]:
def wrong_no_n_check(nums):
    T = reduce(xor, nums)
    if T == 0:
        return 0
    vals = set(nums)
    if len(vals) == 1:
        return -1
    dist, q = {0: 0}, deque([0])
    while q:
        u = q.popleft()
        if u == T:
            return dist[u]                      # SAI: thiếu kiểm tra d == n
        for v in vals:
            if u ^ v not in dist:
                dist[u ^ v] = dist[u] + 1
                q.append(u ^ v)


print("Sai :", wrong_no_n_check([1, 2, 4]), "| Đúng:", sol.minOperations([1, 2, 4]), "| Brute:", min_ops_bruteforce([1, 2, 4]))

Sai : 3 | Đúng: -1 | Brute: -1


## Lỗi 2 — So `d` với số giá trị **phân biệt** thay vì `len(nums)`

Các phần tử trùng nhau là "ô nháp" hợp lệ, nên chỉ `-1` khi `d` bằng **độ dài mảng**.

In [14]:
def wrong_compare_distinct(nums):
    r = sol.minOperations(nums)
    d = r if r != -1 else None
    vals = set(nums)
    if d is not None and d == len(vals):        # SAI: so với số giá trị phân biệt
        return -1
    return r


arr = [1, 2, 4, 1, 1]
print("Sai :", wrong_compare_distinct(arr), "| Đúng:", sol.minOperations(arr), "| Brute:", min_ops_bruteforce(arr))

Sai : -1 | Đúng: 3 | Brute: 3


## Lỗi 3 — Nghĩ rằng thao tác XOR tổng với phần tử **bị thay**

Nhầm `T_mới = T ^ nums[i]` (phần tử bị thay) dẫn tới mô hình sai. Hãy luôn tính lại: `T ^ nums[i] ^ (nums[i] ^ nums[j]) = T ^ nums[j]`.

## Lỗi 4 — BFS trên trạng thái mảng với `n = 10^5`

Số trạng thái khổng lồ. Phải tìm bất biến (tổng XOR) để giảm còn 2048 đỉnh.

## Lỗi 5 — Tham lam "lấy giá trị có bit cao nhất của T"

XOR không cộng dồn như số học; tham lam theo bit dễ cho số bước lớn hơn tối ưu. Luôn dùng BFS (đồ thị không trọng số) để có số bước ít nhất.

In [15]:
def wrong_greedy_bits(nums):
    T = reduce(xor, nums)
    if T == 0:
        return 0
    vals = sorted(set(nums), reverse=True)
    steps, cur, used = 0, T, set()
    while cur:
        hb = cur.bit_length() - 1
        cand = [v for v in vals if (v >> hb) & 1 and v not in used]
        if not cand:
            return -1
        v = min(cand, key=lambda c: c ^ cur)     # chọn giá trị làm cur nhỏ nhất
        used.add(v)
        cur ^= v
        steps += 1
    return -1 if steps == len(nums) else steps


random.seed(5)
for _ in range(20000):
    arr = [random.randint(1, 15) for _ in range(random.randint(3, 6))]
    if wrong_greedy_bits(arr) != sol.minOperations(arr):
        print("Phản ví dụ:", arr, "| tham lam:", wrong_greedy_bits(arr), "| đúng:", sol.minOperations(arr))
        break

Phản ví dụ: [3, 15, 14, 13, 7] | tham lam: -1 | đúng: 2


# 13. Mẹo nhận diện

1. Thao tác phức tạp trên mảng, nhưng câu hỏi chỉ về **một đại lượng tổng hợp** (tổng, XOR, tích...) → tính xem thao tác làm đại lượng đó thay đổi thế nào (**bất biến / biến đổi**).
2. Giá trị nhỏ (`<= 2000`) + XOR → không gian trạng thái chỉ `2^11` → **BFS / DP trên giá trị**.
3. "Ít phần tử nhất có XOR bằng `x`" → BFS trên XOR (đồ thị Cayley).
4. "Không bao giờ về 0" với XOR → nghĩ tới **độc lập tuyến tính trên GF(2)**, cơ sở XOR (XOR basis / Gauss).

Các bài liên quan: *Minimum Number of Operations to Make Array XOR Equal to K* (LeetCode 2997), *Maximum XOR of Two Numbers* (421), *Shortest Path Visiting All Nodes* (847 — BFS trên trạng thái bitmask).

# 14. Biến thể

## Biến thể 1 — Đưa tổng XOR về `K` thay vì `0`

Cần `v1 ^ ... ^ vd = T ^ K`: BFS từ `0` tới `T ^ K`.

## Biến thể 2 — Có tồn tại tập con XOR = 0?

Đúng bài "phụ thuộc tuyến tính": dùng cơ sở XOR (`has_zero_subset` ở mục 5). (Bài tập 5.)

# 15. Template tư duy

```python
# BFS trên không gian giá trị nhỏ (XOR)
dist = [-1] * (1 << BITS); dist[0] = 0
queue = deque([0])
while queue:
    u = queue.popleft()
    for v in distinct_values:
        w = u ^ v
        if dist[w] == -1:
            dist[w] = dist[u] + 1
            queue.append(w)
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

`nums = [5, 6, 9, 9]`. Tính `T`, tìm `d` và đáp án. Viết một dãy thao tác cụ thể. Kiểm tra bằng cell dưới.

In [16]:
# Kiểm tra Bài 1
nums = [5, 6, 9, 9]
print("T =", reduce(xor, nums), "| đáp án =", sol.minOperations(nums))
arr = apply_op(nums, 2, 0)        # ô 2 làm nháp, giữ 5
arr = apply_op(arr, 2, 1)         # giữ 6
print(arr, reduce(xor, arr))

T = 3 | đáp án = 2
[5, 6, 10, 9] 0


## Bài 2 — Dự đoán output

```python
print(sol.minOperations([7, 7]))
print(sol.minOperations([7, 7, 7]))
print(sol.minOperations([1, 2]))
print(sol.minOperations([3, 3, 1, 2]))
```

In [17]:
print(sol.minOperations([7, 7]))
print(sol.minOperations([7, 7, 7]))
print(sol.minOperations([1, 2]))
print(sol.minOperations([3, 3, 1, 2]))

0
-1
-1
1


## Bài 3 — Kiểm chứng công thức `T_mới = T ^ nums[j]`

Viết hàm thử **mọi** thao tác hợp lệ trên một mảng và kiểm tra công thức.

In [18]:
def exercise_3_check_formula(arr: List[int]) -> bool:
    # TODO: với mọi i != j, arr[i] != arr[j], kiểm tra xor(apply_op(arr, i, j)) == xor(arr) ^ arr[j]
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_check_formula([8, 1, 4, 8, 2])

## Bài 4 — Ít phần tử nhất có XOR bằng `target`

Cho tập giá trị `vals` (mỗi giá trị `< 1024`) và `target`. Trả về số phần tử ít nhất (mỗi giá trị dùng tối đa 1 lần) có XOR bằng `target`, hoặc `-1`. (`target = 0` → 0.)

```python
min_xor_count([1, 2, 4, 8], 7)   -> 3
min_xor_count([3, 5], 1)          -> -1
```

In [19]:
def exercise_4_min_xor_count(vals: List[int], target: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_min_xor_count([1, 2, 4, 8], 7) == 3

## Bài 5 — Có tập con khác rỗng XOR = 0?

Trả về `True/False` trong `O(n · 11)` bằng cơ sở XOR.

In [20]:
def exercise_5_has_zero_subset(nums: List[int]) -> bool:
    # TODO
    raise NotImplementedError


# assert exercise_5_has_zero_subset([1, 2, 3]) is True
# assert exercise_5_has_zero_subset([1, 2, 4]) is False

## Bài 6 — Mini challenge: XOR lớn nhất của một tập con

Tìm XOR lớn nhất của một tập con (có thể rỗng) của `nums`, dùng cơ sở XOR.

```python
max_subset_xor([2, 4, 5])   -> 7
```

In [21]:
def exercise_6_max_subset_xor(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: hai vòng lặp `i`, `j`.
- Mức 2: bỏ qua khi `i == j` hoặc `arr[i] == arr[j]`.
- Mức 3: `reduce(xor, ...)`.

**Bài 4**

- Mức 1: BFS từ `0` trên các giá trị `0..1023`.
- Mức 2: cạnh `u -> u ^ v`.
- Mức 3: dùng lại một giá trị thì triệt tiêu, nên BFS không bao giờ chọn nó để có đường ngắn nhất.

**Bài 5**

- Mức 1: giữ danh sách `basis`.
- Mức 2: rút gọn `x` bằng `x = min(x, x ^ b)` cho từng `b` (sắp `basis` giảm dần cũng được).
- Mức 3: nếu `x` về 0 → có phụ thuộc.

**Bài 6**

- Mức 1: xây cơ sở theo bit cao nhất (`basis[bit]`).
- Mức 2: duyệt bit từ cao xuống: `res = max(res, res ^ basis[bit])`.
- Mức 3: đối chiếu brute force mọi tập con.

# 17. Đáp án tham khảo

## Đáp án Bài 1

`T = 5 ^ 6 ^ 9 ^ 9 = 3`. Không giá trị nào bằng 3; `5 ^ 6 = 3` → `d = 2 < 4` → **2**. Dùng ô 2 (một số `9`) làm nháp, giữ `5` rồi giữ `6`: `[5, 6, 9, 9] → [5, 6, 12, 9] → [5, 6, 10, 9]`, tổng XOR `5 ^ 6 ^ 10 ^ 9 = 0`.

## Đáp án Bài 2

```text
0
-1
-1
1
```

- `[7, 7]`: `T = 0`.
- `[7, 7, 7]`: `T = 7`, không thao tác nào hợp lệ.
- `[1, 2]`: `T = 3`, cần cả `{1, 2}` mà `n = 2` → độc lập → `-1`.
- `[3, 3, 1, 2]`: `T = 1 ^ 2 = 3`, mà `3` có trong mảng → `d = 1`.

In [22]:
# Đáp án Bài 3
def answer_3_check_formula(arr: List[int]) -> bool:
    T = reduce(xor, arr)
    for i in range(len(arr)):
        for j in range(len(arr)):
            if i != j and arr[i] != arr[j]:
                if reduce(xor, apply_op(arr, i, j)) != T ^ arr[j]:
                    return False
    return True


assert answer_3_check_formula([8, 1, 4, 8, 2])
random.seed(3)
assert all(answer_3_check_formula([random.randint(1, 50) for _ in range(6)]) for _ in range(200))
print("Bài 3: PASS")

Bài 3: PASS


In [23]:
# Đáp án Bài 4
def answer_4_min_xor_count(vals: List[int], target: int) -> int:
    dist = [-1] * 1024
    dist[0] = 0
    q = deque([0])
    vals = set(vals)
    while q:
        u = q.popleft()
        for v in vals:
            if dist[u ^ v] == -1:
                dist[u ^ v] = dist[u] + 1
                q.append(u ^ v)
    return dist[target]


assert answer_4_min_xor_count([1, 2, 4, 8], 7) == 3
assert answer_4_min_xor_count([3, 5], 1) == -1
assert answer_4_min_xor_count([3, 5], 0) == 0
print("Bài 4: PASS")

Bài 4: PASS


In [24]:
# Đáp án Bài 5
def answer_5_has_zero_subset(nums: List[int]) -> bool:
    return has_zero_subset(nums)


def brute_zero_subset(nums):
    return any(reduce(xor, c) == 0 for s in range(1, len(nums) + 1) for c in combinations(nums, s))


assert answer_5_has_zero_subset([1, 2, 3]) is True
assert answer_5_has_zero_subset([1, 2, 4]) is False
random.seed(55)
for _ in range(500):
    arr = [random.randint(1, 15) for _ in range(random.randint(1, 5))]
    assert answer_5_has_zero_subset(arr) == brute_zero_subset(arr), arr
print("Bài 5: PASS")

Bài 5: PASS


In [25]:
# Đáp án Bài 6
def answer_6_max_subset_xor(nums: List[int]) -> int:
    basis = [0] * 32
    for x in nums:
        for bit in range(31, -1, -1):
            if not (x >> bit) & 1:
                continue
            if basis[bit] == 0:
                basis[bit] = x
                break
            x ^= basis[bit]
    res = 0
    for bit in range(31, -1, -1):
        res = max(res, res ^ basis[bit])
    return res


assert answer_6_max_subset_xor([2, 4, 5]) == 7
random.seed(6)
for _ in range(500):
    arr = [random.randint(0, 31) for _ in range(random.randint(1, 6))]
    best = max([0] + [reduce(xor, c) for s in range(1, len(arr) + 1) for c in combinations(arr, s)])
    assert answer_6_max_subset_xor(arr) == best, arr
print("Bài 6: PASS (khớp brute force)")

Bài 6: PASS (khớp brute force)


# 18. Tổng kết

**Minimum Operations to Make XOR of All Elements Zero**:

1. Bất biến: mỗi thao tác XOR tổng với phần tử **được giữ**.
2. Quy về: số ít nhất các giá trị phân biệt có XOR = `T` — BFS trên 2048 giá trị.
3. `-1` khi: mọi phần tử bằng nhau (không thao tác được), hoặc phải dùng cả mảng (`d == n` ⇔ mảng độc lập tuyến tính trên GF(2) — thao tác giữ nguyên độc lập).

## Độ phức tạp

```text
Time:  O(n + 2048 · D)      (D = số giá trị phân biệt)
Space: O(2048)
```

# 19. Checklist trước khi submit

- [ ] `T == 0` → 0 ngay?
- [ ] Một giá trị duy nhất (và `T != 0`) → `-1`?
- [ ] So `d` với `len(nums)` (không phải số giá trị phân biệt)?
- [ ] BFS chỉ trên các giá trị phân biệt (tránh lặp 10^5 lần mỗi đỉnh)?
- [ ] Đã test: `[1, 2, 4]`, `[7, 7, 7]`, `[1, 2, 4, 1, 1]`?